# Prometheus - Medical Image Segmentation Training

**Models:** UNetTissue | DualUNet (ConvNeXt-v2 U-Net + Transformer + Sparse MoE)

**GPU:** Google Colab G4 / A100 100GB VRAM

---
## 1. Setup

In [ ]:
import os
import sys
import math
import random
from pathlib import Path
from datetime import datetime
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

print(f"PyTorch {torch.__version__}, CUDA {torch.version.cuda}")
print(f"Device count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  [{i}] {torch.cuda.get_device_name(i)}")
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Mount Google Drive (for datasets and checkpoints)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo (if not already present)
REPO_DIR = "/content/prometheus"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hoangtung386/Prometheus.git {REPO_DIR}

%cd {REPO_DIR}
!pip install -e .
!pip install tensorboard matplotlib

In [ ]:
sys.path.insert(0, REPO_DIR)

from prometheus import UNetTissue, DualUNet, CombinedLoss, DiceLoss, FocalLoss, BCEWithLogitsLoss
from prometheus.config import ModelConfig, TrainingConfig
from prometheus.blocks import ConvNeXtBlock, DecoderBlock, LocalGlobalAttention, SparseMoE, EncoderTransformerBlock, EncoderTransformerStack
from prometheus.utils import LayerNorm, GRN

print("All imports OK")

---
## 2. Reproducibility

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

---
## 3. Configuration

In [ ]:
class Config:
    # ── Model ──
    model_type: str = "DualUNet"           # "UNetTissue" or "DualUNet"
    in_chans: int = 3
    num_classes: int = 1
    image_size: int = 256

    # ── Training ──
    batch_size: int = 32                   # 100GB VRAM → lớn
    epochs: int = 200
    lr: float = 2e-4
    weight_decay: float = 1e-2
    warmup_epochs: int = 10
    grad_clip_norm: float = 1.0
    amp: bool = True                       # mixed precision

    # ── Data ──
    data_root: str = "/content/drive/MyDrive/prometheus_data"
    num_workers: int = 8
    pin_memory: bool = True
    val_split: float = 0.1

    # ── Logging / Checkpoint ──
    log_dir: str = "/content/drive/MyDrive/prometheus_logs"
    ckpt_dir: str = "/content/drive/MyDrive/prometheus_checkpoints"
    log_interval: int = 50
    save_interval: int = 5                 # epochs

cfg = Config()
os.makedirs(cfg.log_dir, exist_ok=True)
os.makedirs(cfg.ckpt_dir, exist_ok=True)

---
## 4. Dataset (Yêu cầu tự chuẩn bị)

Thay `YourDataset` bằng loader thật của bạn.
Cấu trúc thư mục gợi ý:
```
data_root/
├── images/      # *.png / *.tif
└── masks/       # cùng tên, 1 channel
```

In [ ]:
class SegmentationDataset(Dataset):
    """Medical image segmentation dataset.

    Expected layout:
        data_root/
        ├── images/    (input, 3-channel PNG)
        └── masks/     (target, 1-channel PNG, values 0/1)
    """
    def __init__(self, root: str, image_size: int = 256, train: bool = True):
        super().__init__()
        self.root = Path(root)
        self.image_size = image_size
        self.train = train

        img_dir = self.root / "images"
        msk_dir = self.root / "masks"
        self.files = sorted([
            p for p in img_dir.iterdir()
            if p.suffix in (".png", ".tif", ".jpg", ".jpeg")
        ])
        self.files = [
            p for p in self.files if (msk_dir / p.name).exists()
        ]
        print(f"Found {len(self.files)} image-mask pairs")

    def __len__(self) -> int:
        return len(self.files)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        img_path = self.files[idx]
        msk_path = self.root / "masks" / img_path.name

        img = Image.open(img_path).convert("RGB")
        msk = Image.open(msk_path).convert("L")

        # Resize
        img = img.resize((self.image_size, self.image_size), Image.BILINEAR)
        msk = msk.resize((self.image_size, self.image_size), Image.NEAREST)

        # Augment (simple flip)
        if self.train and random.random() > 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            msk = msk.transpose(Image.FLIP_LEFT_RIGHT)

        img = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0
        msk = torch.from_numpy(np.array(msk)).unsqueeze(0).float() / 255.0
        msk = (msk > 0.5).float()
        return img, msk


# ── Load data ──
if os.path.exists(cfg.data_root):
    full_dataset = SegmentationDataset(cfg.data_root, cfg.image_size, train=True)
    val_len = int(len(full_dataset) * cfg.val_split)
    train_len = len(full_dataset) - val_len
    train_dataset, val_dataset = random_split(
        full_dataset, [train_len, val_len],
        generator=torch.Generator().manual_seed(42),
    )
    train_loader = DataLoader(
        train_dataset, batch_size=cfg.batch_size, shuffle=True,
        num_workers=cfg.num_workers, pin_memory=cfg.pin_memory,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=cfg.batch_size, shuffle=False,
        num_workers=cfg.num_workers, pin_memory=cfg.pin_memory,
    )
    print(f"Train: {len(train_dataset)} samples, Val: {len(val_dataset)} samples")
else:
    print(f"Data root {cfg.data_root} not found — using dummy data")
    train_loader = None
    val_loader = None

---
## 5. Model & Optimizer

In [ ]:
def build_model(cfg: Config) -> nn.Module:
    model_cfg = ModelConfig(
        in_chans=cfg.in_chans,
        num_classes=cfg.num_classes,
        drop_path_rate=0.1,
    )
    if cfg.model_type == "UNetTissue":
        model = UNetTissue(config=model_cfg)
    elif cfg.model_type == "DualUNet":
        model = DualUNet(config=model_cfg)
    else:
        raise ValueError(f"Unknown model: {cfg.model_type}")
    return model.to(device)


model = build_model(cfg)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{cfg.model_type}: {total_params:,} total / {trainable_params:,} trainable params")

In [ ]:
optimizer = optim.AdamW(
    model.parameters(),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
    betas=(0.9, 0.999),
    eps=1e-8,
)

# Cosine LR with linear warmup
def warmup_cosine_lr(epoch: int) -> float:
    if epoch < cfg.warmup_epochs:
        return epoch / max(1, cfg.warmup_epochs)
    progress = (epoch - cfg.warmup_epochs) / max(1, cfg.epochs - cfg.warmup_epochs)
    return 0.5 * (1 + math.cos(math.pi * progress))


scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_cosine_lr)

# Loss
criterion = CombinedLoss(bce_weight=1.0, dice_weight=1.0)
print("Optimizer, scheduler, criterion ready")

In [ ]:
# GradScaler for AMP (mixed precision)
scaler = torch.cuda.amp.GradScaler(enabled=cfg.amp)

writer = SummaryWriter(log_dir=cfg.log_dir)
print(f"TensorBoard: {cfg.log_dir}")

---
## 6. Training Loop

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: optim.Optimizer,
    criterion: nn.Module,
    scaler: torch.cuda.amp.GradScaler,
    epoch: int,
    cfg: Config,
) -> float:
    model.train()
    total_loss = 0.0
    num_batches = len(loader)

    for batch_idx, (images, masks) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=cfg.amp):
            if cfg.model_type == "UNetTissue":
                logits = model(images)
                loss = criterion(logits, masks)
            else:  # DualUNet
                t_mask, n_mask, moe_loss = model(images)
                loss_t = criterion(t_mask, masks)
                loss_n = criterion(n_mask, masks)
                loss = loss_t + loss_n + 0.01 * moe_loss

        scaler.scale(loss).backward()
        if cfg.grad_clip_norm > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        if batch_idx % cfg.log_interval == 0:
            lr_now = optimizer.param_groups[0]["lr"]
            print(f"E{epoch:03d} B{batch_idx:04d}/{num_batches}  loss={loss.item():.4f}  lr={lr_now:.2e}")

    return total_loss / num_batches


@torch.no_grad()
def validate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    cfg: Config,
) -> tuple[float, float]:
    model.eval()
    total_loss = 0.0
    dice_scores = []

    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        if cfg.model_type == "UNetTissue":
            logits = model(images)
            loss = criterion(logits, masks)
            probs = torch.sigmoid(logits)
        else:
            t_mask, n_mask, _ = model(images)
            loss = criterion(t_mask, masks) + criterion(n_mask, masks)
            probs = torch.sigmoid(n_mask)

        total_loss += loss.item()

        # Dice score
        preds = (probs > 0.5).float()
        intersection = (preds * masks).sum(dim=(1, 2, 3))
        cardinality = preds.sum(dim=(1, 2, 3)) + masks.sum(dim=(1, 2, 3))
        dice = (2 * intersection + 1e-6) / (cardinality + 1e-6)
        dice_scores.append(dice)

    avg_loss = total_loss / len(loader)
    avg_dice = torch.cat(dice_scores).mean().item()
    return avg_loss, avg_dice

In [ ]:
# ── Main training loop ──

best_dice = 0.0
start_epoch = 0

if train_loader is None:
    # Dummy training demo
    print("No data found — running dummy training for 5 epochs")
    dummy_loader = DataLoader(
        [(torch.randn(3, cfg.image_size, cfg.image_size),
          torch.randint(0, 2, (1, cfg.image_size, cfg.image_size)).float())
         for _ in range(64)],
        batch_size=cfg.batch_size, shuffle=True,
    )
    train_loader, val_loader = dummy_loader, dummy_loader
    cfg.epochs = 5

for epoch in range(start_epoch, cfg.epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler, epoch, cfg)
    scheduler.step()

    val_loss, val_dice = validate(model, val_loader, criterion, cfg)

    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar("Loss/val", val_loss, epoch)
    writer.add_scalar("Dice/val", val_dice, epoch)
    writer.add_scalar("LR", optimizer.param_groups[0]["lr"], epoch)

    print(f"─── E{epoch:03d}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_dice={val_dice:.4f}  lr={optimizer.param_groups[0]['lr']:.2e}")

    # Save best
    if val_dice > best_dice:
        best_dice = val_dice
        ckpt_path = os.path.join(cfg.ckpt_dir, f"{cfg.model_type}_best.pth")
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_dice": best_dice,
            "config": cfg,
        }, ckpt_path)
        print(f"  → Saved best model (dice={val_dice:.4f})")

    # Periodic checkpoint
    if (epoch + 1) % cfg.save_interval == 0:
        ckpt_path = os.path.join(cfg.ckpt_dir, f"{cfg.model_type}_epoch{epoch:03d}.pth")
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
        }, ckpt_path)
        print(f"  → Saved checkpoint @ epoch {epoch}")

writer.close()
print(f"\nDone! Best val Dice: {best_dice:.4f}")

---
## 7. Inference & Visualization

In [ ]:
@torch.no_grad()
def predict(model: nn.Module, image: torch.Tensor, cfg: Config) -> torch.Tensor:
    model.eval()
    image = image.unsqueeze(0).to(device)
    if cfg.model_type == "UNetTissue":
        logits = model(image)
        return torch.sigmoid(logits).squeeze(0).cpu()
    else:
        t_mask, n_mask, _ = model(image)
        return torch.sigmoid(n_mask).squeeze(0).cpu()


def show_sample(model, dataset, idx=0):
    img, msk = dataset[idx]
    pred = predict(model, img, cfg)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img.permute(1, 2, 0))
    axes[0].set_title("Input")
    axes[1].imshow(msk.squeeze(0), cmap="gray")
    axes[1].set_title("Ground Truth")
    axes[2].imshow(pred.squeeze(0), cmap="gray")
    axes[2].set_title("Prediction")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


if val_loader is not None:
    show_sample(model, val_dataset)
else:
    print("No validation data to show")

---
## 8. Resume Training từ Checkpoint

In [ ]:
def load_checkpoint(ckpt_path: str, model: nn.Module, optimizer: optim.Optimizer, scheduler, scaler):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if "scaler_state_dict" in ckpt:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_dice = ckpt.get("best_dice", 0.0)
    print(f"Resumed from epoch {ckpt['epoch']} (best dice={best_dice:.4f})")
    return start_epoch, best_dice


# Usage (uncomment to use):
# start_epoch, best_dice = load_checkpoint(
#     "/content/drive/MyDrive/prometheus_checkpoints/DualUNet_best.pth",
#     model, optimizer, scheduler, scaler,
# )

---
## 9. TensorBoard (trong Colab)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {cfg.log_dir}